In [111]:
import io
import fitz
import torch
import transformers
from PIL import Image

from huggingface_hub.utils import enable_progress_bars
from colpali_engine.models import ColQwen2, ColQwen2Processor

import psycopg2
from pgvector.psycopg2 import register_vector

from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption

In [93]:
pdf_path = "pdfs\supplier_guide_detailed.pdf"

In [83]:
transformers.utils.logging.set_verbosity_info()
enable_progress_bars()

In [84]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.mode = "accurate"

In [94]:
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

In [95]:
result = converter.convert(pdf_path)
doc = result.document

[INFO] 2026-04-15 14:14:15,757 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-15 14:14:15,758 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-15 14:14:15,785 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-15 14:14:15,786 [RapidOCR] main.py:50: Using C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-15 14:14:17,803 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-15 14:14:17,805 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-15 14:14:17,807 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-15 14:14:17,807 [RapidOCR] main.py:50: Using C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\

In [96]:
def sanitize_string(s: str) -> str:
    if s is None:
        return None
    return s.replace('\x00', '')

In [98]:
md_index = {page_num: sanitize_string(doc.export_to_markdown(page_no=page_num)) for page_num in doc.pages.keys()}

In [99]:
md_index

{1: '## Government Procurement Guide for Suppliers\n\n(Comprehensive Version)',
 2: "## Contents\n\n| 1. About this Guide............................................................................................................................3   |\n|----------------------------------------------------------------------------------------------------------------------------------------------------|\n| 2. Understanding Government Procurement.....................................................................................3                      |\n| \uf0b7 Government Procurement as Business Opportunities ...........................................................3                                    |\n| \uf0b7 Alignment with International Standards................................................................................4                          |\n| \uf0b7 Government Procurement Principles ....................................................................................4  

In [100]:
fitz_doc = fitz.open(pdf_path)

In [ ]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name, 
    dtype=torch.bfloat16, 
    device_map="auto", # hardware configuration
    trust_remote_code=True,
).eval()

loading configuration file config.json from cache at C:\Users\UserAdmin\.cache\huggingface\hub\models--vidore--colqwen2-base\snapshots\9fe8a713422a7cb4ef79ca77a09b381ee2243101\config.json
Model config Qwen2VLConfig {
  "architectures": [
    "ColQwen2"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float32",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "image_token_id": 151655,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 32768,
  "max_window_layers": 28,
  "model_type": "qwen2_vl",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": {
    "mrope_section": [
      16,
      24,
      24
    ],
    "rope_type": "default",
    "type": "default"
  },
  "rope_theta": 1000000.0,
  "sliding_window": 32768,
  "text_config": {
    "_name_or_path": "/lus/home/CT10/cad15443/mfaysse/colpali/models/Qwen2-VL-2B-Instruct",
    "arc

In [ ]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
) 

loading configuration file preprocessor_config.json from cache at C:\Users\UserAdmin\.cache\huggingface\hub\models--vidore--colqwen2-v1.0\snapshots\83a0134c8f274b3688d8dbde26de8a5b109ad8b4\preprocessor_config.json
Image processor Qwen2VLImageProcessorFast {
  "crop_size": null,
  "data_format": "channels_first",
  "default_to_square": true,
  "device": null,
  "disable_grouping": null,
  "do_center_crop": null,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_pad": null,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "Qwen2VLImageProcessorFast",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "input_data_format": null,
  "max_pixels": 602112,
  "merge_size": 2,
  "min_pixels": 3136,
  "pad_size": null,
  "patch_size": 14,
  "processor_class": "ColQwen2Processor",
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "return_tensors": null,
  "size": {
    "long

In [103]:
vector_index = []

In [ ]:
def get_coarse_embedding(embeddings_tensor):
    normalized = embeddings_tensor / embeddings_tensor.norm(dim=-1, keepdim=True) # L2 Normalization for Cosine Similarity
    return normalized.mean(dim=1).squeeze().to(torch.float32).cpu() 

In [ ]:
for i in range(len(fitz_doc)):
    page = fitz_doc.load_page(i)
    pix = page.get_pixmap(dpi=150)
    img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
    
    inputs = processor.process_images(images=[img])
    inputs = {k: v.to(model.device) for k, v in inputs.items()} # Move to GPU

    with torch.no_grad():
        embeddings = model(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True) # L2 Normalization
        coarse_vector = get_coarse_embedding(embeddings)
    
    vector_index.append({
        "page_number": i+1,
        "coarse_vector": coarse_vector.cpu(),
        "page_embeddings": embeddings.cpu()
        
    })


docker exec -it pgvector-db psql -U postgres

In [106]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)

In [107]:
try:     
    with conn.cursor() as cur:
        for item in vector_index:
            page_number = item["page_number"]
            page_md = md_index.get(page_number, "")

            cur.execute(
                '''
                INSERT INTO pages(page_number, coarse_vector, markdown)
                VALUES (%s, %s, %s) RETURNING id;
                ''',
                (page_number, item["coarse_vector"].flatten().tolist(), page_md)
            )

            page_id = cur.fetchone()[0]

            patch_vectors = item["page_embeddings"].squeeze(0).to(torch.float32).cpu().numpy()

            patch_rows = []
            for index, vec in enumerate(patch_vectors):
                patch_rows.append((
                    page_id, 
                    index, 
                    vec.tolist()
                ))

            cur.executemany(
                '''
                INSERT INTO page_patches (page_id, patch_index, patch_embedding)
                VALUES (%s, %s, %s);
                ''',
                patch_rows
            )

        conn.commit()

except Exception as e: 
        print(f"Error: {e}")
        conn.rollback()